# Proyecto final · Olist: del pedido a la reseña

**Equipo:** Molina Campos Noel Amílcar
Esta plantilla solo fija **el orden y los encabezados** (no los borres: se usan para calificar). El código y las decisiones son de ustedes. Pueden agregar todas las celdas que necesiten dentro de cada parte.

Antes de entregar: *Entorno de ejecución › Reiniciar y ejecutar todo*. El notebook debe correr de principio a fin sin errores.

## 0. Conexión

Usen el usuario de solo lectura del curso. Los datos están en el esquema `final`.

In [1]:
import warnings
from getpass import getpass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sqlalchemy.pool import NullPool
from sqlalchemy.engine import URL 

warnings.filterwarnings("ignore")
#motor = create_engine(getpass("Cadena de conexión: "), poolclass=NullPool)

DB_HOST = "aws-0-us-west-2.pooler.supabase.com"
DB_PORT = 6543
DB_NAME = "postgres"
DB_USER = "alumnos_lectura.ashlmfqpqutuoqtcmzqp"
DB_PASSWORD = getpass("Contraseña de solo lectura: ")

url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
                  # el bloque, igual que en la semana 5

motor = create_engine(url, poolclass=NullPool)    # en vez de la línea de getpass

pd.read_sql("SELECT table_name FROM information_schema.tables WHERE table_schema = 'final' ORDER BY 1", motor)

,table_name
0,category_translation
1,customers
2,order_items
3,order_payments
4,order_reviews
5,orders
6,products
7,sellers


## Parte 1 · Datos

Escriban la consulta SQL que produce **una fila por pedido entregado y con reseña** con las 18 columnas del contrato (ver enunciado, sección "Contrato"). Expliquen en Markdown cada decisión: qué hicieron con pedidos de varios artículos, varios pagos y varias reseñas.

In [2]:
# TU CÓDIGO AQUÍ  postgresql+psycopg2://USUARIO: kodigo CONTRASEÑA@HOST:PUERTO/BASE
nombre_tabla = "orders"

# Las primeras filas
pd.read_sql(f"SELECT * FROM final.{nombre_tabla} LIMIT 5", motor)

# Cuántas filas tiene
pd.read_sql(f"SELECT COUNT(*) AS total_filas FROM final.{nombre_tabla}", motor)

,total_filas
0,80450


total_pedidos = pd.read_sql("SELECT COUNT(*) AS total FROM final.orders", motor)

pedidos_entregados = pd.read_sql(
    "SELECT COUNT(*) AS total FROM final.orders WHERE order_status = '...'", motor
)

pedidos_con_fecha_de_entrega = pd.read_sql(
    "SELECT COUNT(*) AS total FROM final.orders "
    "WHERE order_status = '...' AND ... IS NOT NULL", motor
)

pedidos_con_resena = pd.read_sql(
    "SELECT COUNT(*) AS total FROM final.orders "
    "WHERE order_status = '...' AND ... IS NOT NULL "
    "AND order_id IN (SELECT order_id FROM final.order_reviews)", motor
)

In [3]:
consulta_contrato = """
WITH articulos_por_pedido AS (
    SELECT
        articulo.order_id,
        COUNT(*)                           AS n_items,
        COUNT(DISTINCT articulo.seller_id) AS n_vendedores,
        SUM(articulo.price)                AS precio_total,
        SUM(articulo.freight_value)        AS flete_total,
        SUM(producto.product_weight_g)     AS peso_total_g,
        AVG(producto.product_photos_qty)   AS fotos_promedio
    FROM final.order_items AS articulo
    LEFT JOIN final.products AS producto ON producto.product_id = articulo.product_id
    GROUP BY articulo.order_id
),
primer_articulo AS (
    SELECT
        articulo.order_id,
        producto.product_category_name AS categoria,
        vendedor.seller_state          AS estado_vendedor
    FROM final.order_items AS articulo
    LEFT JOIN final.products AS producto ON producto.product_id = articulo.product_id
    LEFT JOIN final.sellers  AS vendedor ON vendedor.seller_id  = articulo.seller_id
    WHERE articulo.order_item_id = 1
),
pago_principal AS (
    SELECT DISTINCT ON (order_id) order_id, payment_type AS tipo_pago
    FROM final.order_payments
    ORDER BY order_id, payment_value DESC, payment_sequential
),
cuotas_por_pedido AS (
    SELECT order_id, MAX(payment_installments) AS cuotas_max
    FROM final.order_payments
    GROUP BY order_id
),
resena_mas_reciente AS (
    SELECT DISTINCT ON (order_id)
        order_id,
        CASE WHEN review_score IN (1, 2) THEN 1 ELSE 0 END AS mala_resena
    FROM final.order_reviews
    ORDER BY order_id, review_answer_timestamp DESC NULLS LAST
)
SELECT
    pedido.order_id,
    pedido.order_purchase_timestamp,
    pedido.order_approved_at,
    pedido.order_delivered_carrier_date,
    pedido.order_delivered_customer_date,
    pedido.order_estimated_delivery_date,
    cliente.customer_state AS estado_cliente,
    resumen_articulos.n_items,
    resumen_articulos.n_vendedores,
    resumen_articulos.precio_total,
    resumen_articulos.flete_total,
    resumen_articulos.peso_total_g,
    resumen_articulos.fotos_promedio,
    articulo_uno.categoria,
    articulo_uno.estado_vendedor,
    pago.tipo_pago,
    cuotas.cuotas_max,
    resena.mala_resena
FROM final.orders AS pedido
JOIN final.customers AS cliente          ON cliente.customer_id = pedido.customer_id
JOIN resena_mas_reciente AS resena       ON resena.order_id = pedido.order_id
LEFT JOIN articulos_por_pedido AS resumen_articulos ON resumen_articulos.order_id = pedido.order_id
LEFT JOIN primer_articulo AS articulo_uno           ON articulo_uno.order_id = pedido.order_id
LEFT JOIN pago_principal AS pago                    ON pago.order_id = pedido.order_id
LEFT JOIN cuotas_por_pedido AS cuotas               ON cuotas.order_id = pedido.order_id
WHERE pedido.order_status = 'delivered'
  AND pedido.order_delivered_customer_date IS NOT NULL
ORDER BY pedido.order_id
"""

pedidos = pd.read_sql(consulta_contrato, motor)

In [4]:
from pathlib import Path

archivo_cache = Path("pedidos_cache.parquet")

if archivo_cache.exists():
    pedidos = pd.read_parquet("pedidos_cache.parquet")
else:
    pedidos = pd.read_sql(consulta_contrato, motor)
    pedidos.to_parquet(archivo_cache, index=False)

Esta celda verifica el contrato. Debe imprimir `True`:

In [5]:
CONTRATO = ["order_id", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
            "order_delivered_customer_date", "order_estimated_delivery_date", "estado_cliente", "n_items",
            "n_vendedores", "precio_total", "flete_total", "peso_total_g", "fotos_promedio", "categoria",
            "estado_vendedor", "tipo_pago", "cuotas_max", "mala_resena"]
print("Contrato completo:", list(pedidos.columns) == CONTRATO, "|", len(pedidos), "pedidos")

Contrato completo: True | 77304 pedidos


In [6]:
pedidos.head()

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,estado_cliente,n_items,n_vendedores,precio_total,flete_total,peso_total_g,fotos_promedio,categoria,estado_vendedor,tipo_pago,cuotas_max,mala_resena
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,RJ,1,1,58.9,13.29,650.0,4.0,cool_stuff,SP,credit_card,2.0,0
1,00018f77f2f0320c557190d7a144bdd3,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,SP,1,1,239.9,19.93,30000.0,2.0,pet_shop,SP,credit_card,3.0,0
2,000229ec398224ef6ca0657da4fc703e,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,MG,1,1,199.0,17.87,3050.0,2.0,moveis_decoracao,MG,credit_card,5.0,0
3,00042b26cf59d7ce69dfabb4e55b4fd9,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,SP,1,1,199.9,18.14,3750.0,1.0,ferramentas_jardim,PR,credit_card,3.0,0
4,00048cc3ae777c65dbb7d2a0634bc1ea,2017-05-15 21:42:34,2017-05-17 03:55:27,2017-05-17 11:05:55,2017-05-22 13:44:35,2017-06-06,MG,1,1,21.9,12.69,450.0,1.0,utilidades_domesticas,SP,boleto,1.0,0


### Limpieza y diccionario de variables

Faltantes, valores raros y qué hicieron con ellos. Tabla con cada variable que crean en `preparacion.py`: nombre, cómo se calcula y qué significa.

In [7]:
# FALTANTES
faltantes_por_columna = pedidos.isna().sum()
porcentaje_de_faltantes = (faltantes_por_columna / len(pedidos) * 100).round(2)

resumen_de_faltantes = pd.DataFrame({
    "faltantes": faltantes_por_columna,
    "porcentaje": porcentaje_de_faltantes,
})
display(resumen_de_faltantes[resumen_de_faltantes["faltantes"] > 0 ].sort_values("faltantes", ascending=False))


# VALORES RAROS
dias_hasta_la_entrega = (
    pedidos["order_delivered_customer_date"] - pedidos["order_purchase_timestamp"]
).dt.days

entregas_fecha_imposible = (dias_hasta_la_entrega < 0).sum()
cantidad_aprobados_antes_compra = (pedidos["order_approved_at"] < pedidos["order_purchase_timestamp"]).sum()
recepcion_transportista_despues_entrega = (pedidos["order_delivered_carrier_date"] > pedidos["order_delivered_customer_date"]).sum()

display(pedidos[["n_items", "precio_total", "flete_total", "peso_total_g", "fotos_promedio", "cuotas_max"]].describe())

display(pedidos["tipo_pago"].value_counts(dropna=False))



,faltantes,porcentaje
categoria,1263,1.63
fotos_promedio,1239,1.60
peso_total_g,16,0.02
order_approved_at,14,0.02
order_delivered_carrier_date,1,0.00
tipo_pago,1,0.00
cuotas_max,1,0.00


,n_items,precio_total,flete_total,peso_total_g,fotos_promedio,cuotas_max
count,77304.000000,77304.000000,77304.000000,77288.000000,76065.000000,77303.000000
mean,1.142994,136.659358,22.318079,2458.953900,2.250641,2.954064
std,0.539446,206.695949,20.504954,4902.493553,1.744648,2.732586
min,1.000000,0.850000,0.000000,2.000000,1.000000,0.000000
25%,1.000000,45.900000,13.790000,300.000000,1.000000,1.000000
50%,1.000000,86.900000,16.790000,800.000000,2.000000,2.000000
75%,1.000000,149.900000,23.700000,2150.000000,3.000000,4.000000
max,21.000000,13440.000000,1002.290000,184400.000000,20.000000,24.000000


tipo_pago
credit_card    58382
boleto         15668
voucher         2457
debit_card       796
NaN                1
Name: count, dtype: int64

In [ ]:
#TABLA DE DECISIONES

problemas_encontrados = [
    {
        "problema": "Pedidos sin peso registrado",
        "cuantos_pedidos": faltantes_por_columna["peso_total_g"],
        "decision": "Rellenar con la mediana y agregar una columna que avise que faltaba",
        "por_que": "Los productos sin peso son pocos y borrar pedidos dejaría al modelo sin esos casos",
    },
    {
        "problema": "Errores de fecha",
        "cuantos_pedidos": entregas_fecha_imposible,
        "decision": "Cambiar los datos a nulos y agregar otra columna que lo explique",
        "por_que": "Este tipo de error termina en casos como la no entrega de un producto",
    },
    {
        "problema": "Fecha de aprobación pre compra",
        "cuantos_pedidos": cantidad_aprobados_antes_compra,
        "decision": "Cambiar la fecha a la fecha de compra más un día",
        "por_que": "Porque se estima como normal que la aprobación sea despúes de la compra",
    },
    {
            "problema": "Error de registro al recibir el transportista",
            "cuantos_pedidos": recepcion_transportista_despues_entrega,
            "decision": "Cambiar la fecha a la fecha de compra más 2 días",
            "por_que": "Porque se estima como normal que el transportista reciba días despúes de la compra",
    },
    # además, una fila por cada columna que SÍ tenga faltantes,
    # con su cifra: faltantes_por_columna["nombre_de_la_columna"]
]

decisiones_de_limpieza = pd.DataFrame(problemas_encontrados)
decisiones_de_limpieza = decisiones_de_limpieza[decisiones_de_limpieza["cuantos_pedidos"] > 0]
display(decisiones_de_limpieza)

,problema,cuantos_pedidos,decision,por_que
0,Pedidos sin peso registrado,16,Rellenar con la mediana y agregar una columna ...,Los productos sin peso son pocos y borrar pedi...
3,Error de registro al recibir el transportista,23,Cambiar la fecha a la fecha de compra más 3 días,Porque se estima como normal tome entre 48 y 7...


## Parte 2 · Modelo de riesgo

Recuerden:

- La validación es **por fecha**.
- Comparen contra la regla "ordenar por días de retraso" usando **precisión en el 10% de mayor riesgo**.
- Las funciones que transforman las columnas del contrato van en `preparacion.py` y se importan aquí.

In [ ]:
from preparacion import crear_variables      # su archivo, en la misma carpeta

from preparacion import crear_variables

variables_de_prueba = crear_variables(pedidos[CONTRATO[:-1]])
display(variables_de_prueba.head())

In [ ]:
# TU CÓDIGO AQUÍ

### Modelo final y prueba del contrato

Entrenen el modelo final con todos los pedidos, guárdenlo como `modelo_riesgo.joblib` y comprueben que, **cargado desde el archivo**, predice con filas que solo traen las columnas del contrato:

In [ ]:
# TU CÓDIGO AQUÍ

In [ ]:
import joblib
cargado = joblib.load("modelo_riesgo.joblib")
cargado.predict_proba(pedidos[CONTRATO[:-1]].head(5))[:, 1]

### Preguntas (respóndanlas en Markdown, con cifras de sus salidas)

1. ¿Cuánto mejora su modelo sobre la regla de retraso? ¿Y sobre el azar?
2. ¿Qué tres variables pesan más y por qué tiene sentido para el negocio?
3. ¿Qué pasaría con su modelo en un periodo con muy pocos retrasos?

## Parte 3 · Pregunta abierta

Opción elegida: **[3A pronóstico de junio / 3B segmentación]**

Justifiquen la elección y cada decisión. Si eligen 3A, guarden `pronostico_junio.csv` con columnas `fecha, pronostico, bajo, alto` (30 filas, del 1 al 30 de junio de 2018).

In [ ]:
# TU CÓDIGO AQUÍ

## Parte 4 · Deep learning: sentimiento de los comentarios

Clasifiquen una muestra de comentarios con un modelo preentrenado de Hugging Face que entienda portugués y compárenlo con las estrellas. Activen la GPU de Colab para que sea rápido.

In [ ]:
!pip install -q transformers

In [ ]:
# TU CÓDIGO AQUÍ

### Preguntas

1. ¿En qué porcentaje coincide el sentimiento con las estrellas? ¿Dónde está la mayor confusión?
2. Muestren al menos 5 desacuerdos y clasifíquenlos: ¿error del modelo o del cliente?
3. ¿Qué dice el texto que las estrellas no dicen?

## Extra (opcional, +5) · Red neuronal contra XGBoost

Misma división por fecha y mismas variables. Muestren la curva de pérdida y expliquen el resultado.

In [ ]:
# TU CÓDIGO AQUÍ

## Parte 5 · Despliegue

- URL de la API en Render: **[pegar aquí]**
- URL de la app de Streamlit: **[pegar aquí]**

Versiones de este entorno (para su `requirements.txt` de la API):

In [ ]:
import sklearn, xgboost
print(f"pandas=={pd.__version__}\nnumpy=={np.__version__}\nscikit-learn=={sklearn.__version__}\nxgboost=={xgboost.__version__}")

Prueba de la API desplegada (debe devolver una probabilidad):

In [ ]:
import requests
URL_API = "https://SU-API.onrender.com"
pedido = pedidos[CONTRATO[:-1]].head(1).astype(object).where(pedidos[CONTRATO[:-1]].head(1).notna(), None).iloc[0].to_dict()
pedido = {k: (str(v) if isinstance(v, pd.Timestamp) else v) for k, v in pedido.items()}
requests.post(URL_API + "/riesgo", json=pedido, timeout=90).json()

## Parte 6 · Ficha del modelo

La ficha va en `ficha_modelo.md` (también se muestra en la app). Aquí, solo el enlace o una copia.

## Uso de IA

Indiquen qué herramientas de IA usaron y para qué. Todo lo que entregan lo deben poder explicar en la defensa.